In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

# **Chain with custom Runnable**

In [3]:
#Task 1 [Prompt]
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief: {input}")
])

#Task 2 [LLm]
llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

#Task 3 [Str Parser]
str_parser = StrOutputParser()

In [6]:
#Task 4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str) -> dict:
    return {"text" : text}

#create a wrapper for the function
dictionary_maker_runnable = RunnableLambda(dictionary_maker)

# **Option 1: Decicated chains**

In [10]:
# Task 1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")
])

#Task 2 [LLm]
llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

#Task 3 [Str Parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser

# **Option 2: function**

In [11]:
def insta_chain(text: dict):

    text = text["text"]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a Instagram post generator"),
        ("human", "Create a post for the following text for Instagram: {text}")
    ])

    #Task 2 [LLm]
    llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

    #Task 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)
    return result

#it's easy to manage

insta_chain_runnable = RunnableLambda(insta_chain)

# **Final Orchestation**

In [12]:
final_chain = (
    prompt_template |
    llm_openai |
    str_parser |
    dictionary_maker_runnable | 
    RunnableParallel(branches = {"linkedin" : chain_linkedin, "instagram": insta_chain_runnable})
)

final_chain.invoke("Up")

{'branches': {'linkedin': 'Watched Up (2009) again this week — and was struck by how much a 96-minute animated film has to teach professionals about leadership, change, and human connection. 🎈\n\nQuick snapshot: widower Carl Fredricksen literally lifts his house toward Paradise Falls to honor a promise to his late wife Ellie. He unintentionally brings along young Wilderness Explorer Russell, and together they meet a giant bird (Kevin), a talking dog (Dug), and the obsessed explorer Charles Muntz. What begins as a solo quest becomes a story about grief, friendship, and learning to let go.\n\nTakeaways for work and life:\n- Honor your commitments, but don’t let them become anchors that keep you from moving forward.\n- Curiosity and eagerness (Russell) can unlock new perspectives—even for those set in their ways.\n- Unexpected collaborators (Kevin, Dug) often bring the skills and joy you didn’t know you needed.\n- Obsession with “being right” (Muntz) can blind you to ethics and relationsh

## **Chain as a Runnable**

In [15]:
#beautify the output
#Task 1 [Beautify Function]
def beautify(final_response : dict) -> dict:

     linkedin_response = final_response["branches"]["linkedin"]
     instagram_response = final_response["branches"]["instagram"]

     return {"linkedin" : linkedin_response, "instagram": instagram_response}
beautify_runnable = RunnableLambda(beautify)

#Task 2 [final Chain]

#final_chain

#Beaufity Chain
beautify_chain = final_chain | beautify_runnable

beautify_chain.invoke("Inception")

{'linkedin': 'Christopher Nolan’s Inception is more than a sci-fi heist—it’s a masterclass in complex projects, teams, and the human factors that shape outcomes.\n\nQuick summary: Dom Cobb and his crew enter layered dream worlds to plant an idea in a target’s mind. As the mission deepens, Cobb’s unresolved grief for his late wife—manifesting inside the dream—threatens the plan and everyone involved. The film blends action with deep questions about reality, identity, and memory, and closes on an intentionally ambiguous note.\n\nThree workplace lessons I take from it:\n- Architect for complexity: build layers deliberately (design, prototype, test) so each level supports the next.  \n- Assemble complementary specialists: tough problems require diverse skills and tight coordination.  \n- Don’t ignore human factors: unresolved personal issues or unacknowledged biases can derail even the best technical plans.\n\nIn short: strategy + team + emotional intelligence = better odds of success. How